# Bronze — ingest sample orders

**Layer:** raw-ish landing. Keep source-shaped data, do not business-clean yet.

Sample rows come from the wheel the bundle builds (`dist/*.whl`), installed on this task.
In a real source, replace that call with API / JDBC / Auto Loader.


In [ ]:
dbutils.widgets.text("env", "dev")
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")

env = dbutils.widgets.get("env")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

if not catalog or not schema:
    raise ValueError("Job parameters catalog and schema are required")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
bronze_table = f"{catalog}.{schema}.example_orders_bronze"


In [ ]:
from datetime import date

from src.example_source.utils.orders import sample_raw_orders

rows = sample_raw_orders(n=20, as_of=date(2026, 8, 17))
bronze_df = spark.createDataFrame(
    rows,
    schema="order_id string, product string, qty int, unit_price double, region string, order_date string, status string",
)

bronze_df.write.mode("overwrite").saveAsTable(bronze_table)
print(f"Wrote {bronze_df.count()} rows to {bronze_table} (env={env})")
bronze_df.show(5, truncate=False)
